https://github.com/miladlink/TinyYoloV2

https://github.com/eriklindernoren/PyTorch-YOLOv3


# Setup (Only use the first time or if the dataset was changed)

In [2]:
!python filter_sample_json.py data/COCO2017/images/valid_sample data/COCO2017/annotations/instances_val2017_modified_sample.json data/COCO2017/images/5000 data/COCO2017/annotations/instances_train2017_modified_sample.json

Found 20 images in the 'data/COCO2017/images/valid_sample' directory.
Successfully loaded original JSON file: 'data/COCO2017/annotations/instances_val2017.json'.
After filtering, 20 image annotations will be kept.
After filtering, 143 annotations will be kept.
Category ids now conform
Filtering complete! The new JSON file has been saved to: 'data/COCO2017/annotations/instances_val2017_modified_sample.json'
Found 1349 images in the 'data/COCO2017/images/5000' directory.
Successfully loaded original JSON file: 'data/COCO2017/annotations/instances_train2017.json'.
After filtering, 1349 image annotations will be kept.
After filtering, 9953 annotations will be kept.
Category ids now conform
Filtering complete! The new JSON file has been saved to: 'data/COCO2017/annotations/instances_train2017_modified_sample.json'


# Libraries

In [2]:
import os
# import time
# from PIL import Image
# import numpy as np
import json
import cv2
from tqdm import tqdm
# import skimage.io as io
# import matplotlib.pyplot as plt
# from pycocotools.coco import COCO
# import torch
import torch.optim as optim
# import torchvision
# from torchvision import transforms
import torchvision.transforms as transforms
from torchvision.datasets.coco import CocoDetection
from torch.utils.data import DataLoader

from utils.YOLOv2 import *
from models.YOLOv3 import load_model
from attacks.FGSM import FGSM
from attacks.PGD import PGD
from attacks.CW import CW
from attacks.noise import Noise
from detect import detect_image
from utils.loss import compute_loss
from utils.utils import load_classes, rescale_boxes, non_max_suppression, print_environment_info
from utils.augmentations import TRANSFORM_TRAIN, TRANSFORM_VAL
from utils.transforms import DEFAULT_TRANSFORMS, Resize, ResizeEval

# Helper functions + vars


In [3]:
def xyxy2xywh(x):
    # Convert nx4 boxes from [x1, y1, x2, y2] to [x, y, w, h] where xy1=top-left, xy2=bottom-right
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[..., 0] = (x[..., 0] + x[..., 2]) / 2  # x center
    y[..., 1] = (x[..., 1] + x[..., 3]) / 2  # y center
    y[..., 2] = x[..., 2] - x[..., 0]  # width
    y[..., 3] = x[..., 3] - x[..., 1]  # height
    return y

def xywh2xyxy(x):
    # Convert nx4 boxes from [x, y, w, h] to [x1, y1, x2, y2] where xy1=top-left, xy2=bottom-right
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[..., 0] = x[..., 0] - x[..., 2] / 2  # top left x
    y[..., 1] = x[..., 1] - x[..., 3] / 2  # top left y
    y[..., 2] = x[..., 0] + x[..., 2] / 2  # bottom right x
    y[..., 3] = x[..., 1] + x[..., 3] / 2  # bottom right y
    return y

def yolo2json(boxes, img_copy, image_id):
    # * put into coco format of x_min,y_min, width, height, bbox_conf, cls
    # yolo format is x_center, y_center, w, h, bbox_conf, cls_conf, cls
    predictions = []
    for box in boxes:
        x_center, y_center, w, h, conf, cls = box
        x_min = max(0, (x_center - w / 2) * img_copy.shape[3])
        y_min = max(0, (y_center - h / 2) * img_copy.shape[2])
        width = min(img_copy.shape[3], w * img_copy.shape[3])
        height = min(img_copy.shape[2], h * img_copy.shape[2])
        # print(x_min,y_min, width, height, bbox_conf, cls)
        predictions.append({
            'image_id': image_id,
            'category_id': int(id_list[int(cls)]) if modelv == 3 else int(cls),
            'bbox': [int(x_min), int(y_min), int(width), int(height)],
            'score': round(float(conf),2)
        })
    return predictions

def nms2yolo(boxes, img_copy):
    boxes = xyxy2xywh(boxes) # convert from coco to yolo: nms returns nx6 (x1, y1, x2, y2, conf, cls), change to center coordinates [x_center, y_center, width, height]
    boxes[:,0] = boxes[:,0]/img_copy.shape[3]
    boxes[:,1] = boxes[:,1]/img_copy.shape[2]
    boxes[:,2] = boxes[:,2]/img_copy.shape[3]
    boxes[:,3] = boxes[:,3]/img_copy.shape[2]
    return boxes

def saveImageWithBoxes(images, boxes, class_names, fileName):
    to_pil = transforms.ToPILImage()
    pil_image = to_pil(images.squeeze())
    pred_img = plot_boxes(pil_image, boxes, None, class_names)
    pred_img.save(fileName)

def saveImage(img):
    # * just for sanity check, output image. put the dim 3 at the back
    imageN = img.clone().detach()
    imageN = imageN.cpu().squeeze().permute(1, 2, 0).numpy()
    imageN = cv2.cvtColor(imageN, cv2.COLOR_RGB2BGR)
    # print(imageN.shape)
    cv2.imwrite("data/results/mygraph.jpg", imageN*255)

def getOneIter(dataloader):
    images, annotations = next(iter(dataloader))
    np.set_printoptions(linewidth=500)
    np.set_printoptions(suppress=True)
    print("dataloader out")
    print(annotations[0].numpy())


def imgToGreyscale(img):
    if img.shape[0] != 3:
        raise ValueError("Input tensor must have shape [3, H, W].")
    grayscale = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    grayscale_tensor = grayscale.unsqueeze(0).repeat(3, 1, 1)
    return grayscale_tensor

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

os.environ['CUDA_LAUNCH_BLOCKING'] = '1' # reset CUDA debugging environment variable
os.environ['TORCH_USE_CUDA_DSA'] = '1' # enable CUDA DSA for debugging

In [5]:
epochs = 100 # currently, 100 seems like it works very well
checkpoint_interval = 50 
# if time is limited then make this smaller, do note that checkpoints are around 270MB per.
modelv = 3
img_size=416

root_train = "./data/COCO2017/images/1000"
annFile_train = "./data/COCO2017/annotations/instances_train2017_modified_sample.json"
root_val = "./data/COCO2017/images/valid_sample"
annFile_val = "./data/COCO2017/annotations/instances_val2017_modified_sample.json"

mode = "image" # need different modes if i want to save image or output prediction json
# mode = "json"
image_ids= [139, 285, 632, 724, 776, 785, 802, 872, 885, 1000,
            1268, 1296, 1353,1425, 1490, 1503, 1532, 1584, 1675, 1761]

In [6]:
class_names = ['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
id_list = np.array(range(0,80))

# Model import

In [7]:
# if modelv == 2:
#     model = load_model_v2(weights = './weights/yolov2-tiny-voc.weights').to(device)
#     class_names = ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'TVmonitor']
#     root_train = "./data/VOC2007/JPEGImages"
#     annFile_train = "./data/VOC2007/annotations/train.json"
#     root_val = "./data/VOC2007/JPEGImages"
#     annFile_val = "./data/VOC2007/annotations/val.json"

if modelv == 3:
    model = load_model("./config/yolov3.cfg", "./weights/yolov3.weights")

else:
    print("invalid model number!")

# COCO loader

create dataloader (make different train and val later)

In [20]:
# coco_dataset_train = CocoDetection(root=root_train, annFile=annFile_train, transform=TRANSFORM_TRAIN_IMG, target_transform=TRANSFORM_TRAIN_TARGET)
# coco_dataset_train = CocoDetection(root=root_train, annFile=annFile_train, transforms=TRANSFORM_TRAIN)
coco_dataset_train = CocoDetection(root=root_train, annFile=annFile_train, transforms=TRANSFORM_TRAIN)
coco_dataset_val = CocoDetection(root=root_val, annFile=annFile_val, transforms=TRANSFORM_VAL)
# coco_dataset_eval = CocoDetection(root=root_val, annFile=annFile_val, transform=transforms.Compose([transforms.ToTensor(),]))

def collate_fn(batch):
    return tuple(zip(*batch))

# Create a DataLoader for your COCO dataset
train_loader = DataLoader(coco_dataset_train, batch_size=32, shuffle=True, collate_fn=collate_fn) # multiple images per batch
val_loader = DataLoader(coco_dataset_val, batch_size=1, shuffle=True, collate_fn=collate_fn)
# one per batch
# cocoeval_loader = DataLoader(coco_dataset_eval, batch_size=1, shuffle=True, collate_fn=collate_fn) # original images without transformatios


loading annotations into memory...
Done (t=0.14s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [9]:
getOneIter(train_loader) # print targets
getOneIter(val_loader) # print targets

dataloader out
[[2139.            1.            0.19030066    0.165625      0.80969934    0.65319915]
 [2139.           25.            0.07975445    0.165625      0.124546      0.26003118]
 [2139.           32.            0.90532713    0.37408109    0.04956865    0.04411223]
 [2139.           59.            0.17002554    0.165625      0.82997446    0.64514799]
 [2139.           73.            0.29529333    0.2160374     0.04169645    0.07265801]]
dataloader out
[[1761.            4.            0.60660939    0.21778126    0.08003125    0.06840625]
 [1761.            4.            0.40026562    0.01348438    0.11576562    0.12060937]
 [1761.            0.            0.18376562    0.96243753    0.0075625     0.01804686]
 [1761.            0.            0.1723125     0.96304684    0.01128125    0.01529694]
 [1761.            0.            0.18996875    0.96089058    0.00973437    0.02007818]
 [1761.            0.            0.22225       0.97771873    0.0075        0.01131248]
 [1761.     

# Adversarial training

In [10]:
eps = 0.05
# attacker = FGSM(model=model, epsilon=0.05)
# attacker = PGD(model=model, epsilon=0.05, epoch=5, lr=0.02)
attacker = CW(model=model, epsilon=eps, lr=eps/3, epoch=5, target=52) # 52 is banana
# attacker = Noise(model=model, epsilon=0.1)


In [11]:
losses = []
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(
            params,
            lr=model.hyperparams['learning_rate'],
            weight_decay=model.hyperparams['decay'],
        )
min_loss = 100

for epoch in range(1, epochs+1):
    print(f"Starting epoch {epoch}")
    lossesEpoch = []

    for batch_idx, (images, targets) in enumerate(tqdm(train_loader)):
        
        model.train()

        if targets[0].numel() != 0:
            try:
                #* modify inputs to be in proper shape
                images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
                images = images.to(device)

                # modify targets to be in proper shape
                for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
                    if boxes.ndim == 2:
                        boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss

                targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
                targets = targets[:, :6]
                # class indices are mixed up prior

                # # verify class indices range
                # class_indices = targets[:, 1].long()
                # valid_classes = (class_indices >= 0) & (class_indices < 80)

                # if not valid_classes.all():
                #     print(f"Warning: Invalid class indices found: {class_indices[~valid_classes]}")
                #     # Filter out invalid classes
                #     targets = targets[valid_classes]
                #     if targets.shape[0] == 0:
                #         print("No valid targets after filtering, skipping batch")
                #         continue

                # ensure all class indices are long
                targets[:, 1] = targets[:, 1].long()

                print(f"Batch {batch_idx}: targets shape: {targets.shape}, class range: {targets[:, 1].min()}-{targets[:, 1].max()}")

                images_adv = attacker.forward(images, targets) # get adversarial image
                outputsBefore = model(images)
                lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
                outputsAfter = model(images_adv)
                lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
                loss = lossBefore + lossAfter

                lossesEpoch.append(loss.detach().cpu().numpy())
                loss.backward()
                optimizer.step()
                # Reset gradients
                optimizer.zero_grad()

                time.sleep(0.1) # for using noise attack

            except RuntimeError as e:
                print(f"Error in batch {batch_idx}: {e}")
                print(f"Targets shape: {targets.shape if 'targets' in locals() else 'undefined'}")
                if 'targets' in locals():
                    print(f"Class indices: {targets[:, 1].unique()}")
                # clear gradients and continue to next batch
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                continue

        else:
            continue # pics without targets

    if lossesEpoch:
        losses_avg = np.average(lossesEpoch)
        print(f"Epoch {epoch} average loss: {losses_avg}")
        losses.append(losses_avg)
        if losses_avg < min_loss:
            checkpoint_path = f"./data/results/checkpoints/yolov3_ckpt_best.pth"
            print(f"---- Saving new best checkpoint to: '{checkpoint_path}' ----")
            os.makedirs("./data/results/checkpoints", exist_ok=True)
            torch.save(model.state_dict(), checkpoint_path)
            min_loss = losses_avg

    if epoch % checkpoint_interval == 0:
        checkpoint_path = f"./data/results/checkpoints/yolov3_ckpt_{epoch}.pth"
        print(f"---- Saving checkpoint to: '{checkpoint_path}' ----")
        os.makedirs("./data/results/checkpoints", exist_ok=True)
        torch.save(model.state_dict(), checkpoint_path)
        print(f"The best loss has been {min_loss}")

Starting epoch 1


  0%|          | 0/32 [00:00<?, ?it/s]

Batch 0: targets shape: torch.Size([281, 6]), class range: 0.0-75.0


  3%|▎         | 1/32 [00:02<01:12,  2.33s/it]

Batch 1: targets shape: torch.Size([209, 6]), class range: 0.0-74.0


...

Batch 30: targets shape: torch.Size([235, 6]), class range: 0.0-77.0


 97%|█████████▋| 31/32 [00:49<00:01,  1.60s/it]

Batch 31: targets shape: torch.Size([69, 6]), class range: 0.0-75.0


100%|██████████| 32/32 [00:50<00:00,  1.58s/it]


Epoch 100 average loss: 0.22900725901126862
---- Saving checkpoint to: './data/results/checkpoints/yolov3_ckpt_100.pth' ----


## Load adversarial trained model

In [18]:
if modelv == 3:
    model = load_model("./config/yolov3.cfg", f"./data/results/checkpoints/yolov3_ckpt_best.pth")

else:
    print("invalid model number!")

# Attack Evaluation

In [13]:
# attackImage = 0 # variable for saving attack image, run this first, change pruning ratio (attack), 
# #don't run this and only run below cells

### NOTE: Attacker was defined here.

In [21]:
predictionsBefore = []
predictionsAfter = []
lossesBefore = []
lossesAfter = []

os.makedirs("./data/results/images", exist_ok=True)

for i, (images, targets) in enumerate(tqdm(val_loader)):
    if targets[0].numel() != 0:
        with torch.no_grad():
            #* modify inputs to be in proper shape
            images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
            images = images.to(device)
            image_id = int(targets[0][0,0].cpu().numpy()) # assume 1 image
            if image_id not in image_ids: continue # for when we want outputs of specific images
            for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
                if boxes.ndim == 2: boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss. this is normally done in ListDataset -> collate_fn. the id now starts at 0 for each image
            targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
            # originalImageSize = targets[0, 6:].cpu().numpy() # original image shape, assume one image per batch - NOT available in json format
            img_info = coco_dataset_val.coco.imgs[image_id]
            originalImageSize = (img_info['height'], img_info['width'])
            targets = targets[:, :6]

            # print debugging information
            print(f"Image ID: {image_id}")
            print(f"Original targets shape: {targets.shape}")
            print(f"Targets data:")
            print(targets)
            print(f"Class indices: {targets[:, 1]}")
            print(f"Class range: {targets[:, 1].min()} - {targets[:, 1].max()}")

            # check mapping of class indices
            original_classes = targets[:, 1].clone()
            print(f"Original class IDs: {original_classes}")

            # mapping class indices ([1, 80] to [0, 79] range
            targets[:, 1] = targets[:, 1] - 1

            # verify mapped class indices
            mapped_classes = targets[:, 1]
            print(f"Mapped class IDs: {mapped_classes}")
            print(f"Mapped class range: {mapped_classes.min()} - {mapped_classes.max()}")

            # # ensure all classes are in range [0, 79]
            # valid_mask = (mapped_classes >= 0) & (mapped_classes < 80)
            # if not valid_mask.all():
            #     print(f"Invalid class indices found: {mapped_classes[~valid_mask]}")
            #     targets = targets[valid_mask]
            #     if targets.shape[0] == 0:
            #         print("No valid targets after filtering, skipping image")
            #         continue
            #     print(f"Filtered targets shape: {targets.shape}")

            # ensure all class indices are long
            targets[:, 1] = targets[:, 1].long()

            # final validation
            final_classes = targets[:, 1]
            print(f"Final class indices: {final_classes}")
            print(f"Final class range: {final_classes.min()} - {final_classes.max()}")
            print(f"All classes in range [0, 79]: {((final_classes >= 0) & (final_classes < 80)).all()}")

            #* loss
            model.train()
            try:
                outputsBefore = model(images)
                print(f"Model output shapes: {[out.shape for out in outputsBefore]}")

                lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
                lossesBefore.append(lossBefore.cpu().numpy())

                images_adv = attacker.forward(images, targets) # get adversarial image

                outputsAfter = model(images_adv)
                lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
                lossesAfter.append(lossAfter.cpu().numpy())

            except RuntimeError as e:
                print(f"CUDA error occurred: {e}")
                print(f"Error details:")
                print(f"  Targets shape: {targets.shape}")
                print(f"  Class indices: {targets[:, 1]}")
                # print(f"  Class unique values: {targets[:, 1].unique()}")
                print(f"  Class data type: {targets[:, 1].dtype}")

                # clean CUDA cache and skip this iteration
                torch.cuda.empty_cache()
                continue

            #* plot
            model.eval()

            # before attack
            outputsBefore = model(images[0].unsqueeze(0))
            boxesBefore = non_max_suppression(outputsBefore, conf_thres=0.3, iou_thres=0.5)[0].numpy()
            if mode == "json":
                boxesBefore = rescale_boxes(boxesBefore, img_size, originalImageSize)
            boxesBefore = nms2yolo(boxesBefore, images)
            if mode == "image":
                saveImageWithBoxes(images[0], boxesBefore, class_names, f"./data/results/images/attack_before_{image_id}.jpg")
            if mode == "json":
                predictionsBefore += yolo2json(boxesBefore, images[0].unsqueeze(0), image_id)

            # after attack
            outputsAfter = model(images_adv[0].unsqueeze(0))
            boxesAfter = non_max_suppression(outputsAfter, conf_thres=0.3, iou_thres=0.5)[0].numpy()

            if mode == "json":
                boxesAfter = rescale_boxes(boxesAfter, img_size, originalImageSize)
            boxesAfter = nms2yolo(boxesAfter, images_adv)
            print(boxesAfter)
            if mode == "image":
                saveImageWithBoxes(images_adv[0], boxesAfter, class_names, f"./data/results/images/attack_after_{image_id}.jpg")
            if mode == "json":
                predictionsAfter += yolo2json(boxesAfter, images_adv[0].unsqueeze(0), image_id)

    else: continue # pics without targets

with open(f'./data/results/predictionsBefore.json', 'w') as f:
    json.dump(predictionsBefore, f)
with open(f'./data/results/predictionsAfter.json', 'w') as f:
    json.dump(predictionsAfter, f)
np.savetxt("./data/results/lossesBefore.csv", lossesBefore, delimiter=",")
np.savetxt("./data/results/lossesAfter.csv", lossesAfter, delimiter=",")

  0%|          | 0/20 [00:00<?, ?it/s]

Image ID: 1584
Original targets shape: torch.Size([14, 6])
Targets data:
tensor([[0.0000, 5.0000, 0.2063, 0.1480, 0.6502, 0.7265],
        [0.0000, 0.0000, 0.3110, 0.4980, 0.0970, 0.1074],
        [0.0000, 0.0000, 0.7118, 0.5526, 0.0265, 0.0406],
        [0.0000, 0.0000, 0.4884, 0.5469, 0.0570, 0.0547],
        [0.0000, 0.0000, 0.2850, 0.2783, 0.0480, 0.0375],
        [0.0000, 0.0000, 0.4856, 0.2627, 0.0414, 0.0341],
        [0.0000, 0.0000, 0.1983, 0.6368, 0.0114, 0.0346],
        [0.0000, 5.0000, 0.9285, 0.5178, 0.0715, 0.1439],
        [0.0000, 0.0000, 0.1490, 0.6350, 0.0216, 0.0564],
        [0.0000, 0.0000, 0.3469, 0.2758, 0.0302, 0.0336],
        [0.0000, 0.0000, 0.1276, 0.6170, 0.0313, 0.0861],
        [0.0000, 0.0000, 0.1665, 0.6496, 0.0216, 0.0527],
        [0.0000, 0.0000, 0.1856, 0.6402, 0.0212, 0.0631],
        [0.0000, 5.0000, 0.8218, 0.5163, 0.1032, 0.1139]], device='cuda:0',
       dtype=torch.float64)
Class indices: tensor([5., 0., 0., 0., 0., 0., 0., 5., 0., 0., 0., 0.

  5%|▌         | 1/20 [00:00<00:07,  2.64it/s]

[]
Image ID: 1353
Original targets shape: torch.Size([7, 6])
Targets data:
tensor([[0.0000, 6.0000, 0.2566, 0.6472, 0.3798, 0.2786],
        [0.0000, 0.0000, 0.5532, 0.3090, 0.1330, 0.2582],
        [0.0000, 0.0000, 0.4218, 0.2730, 0.1510, 0.1052],
        [0.0000, 0.0000, 0.4012, 0.3962, 0.2561, 0.3691],
        [0.0000, 0.0000, 0.2482, 0.4277, 0.2334, 0.2858],
        [0.0000, 0.0000, 0.3919, 0.3611, 0.1521, 0.1454],
        [0.0000, 0.0000, 0.5013, 0.3656, 0.0640, 0.1255]], device='cuda:0',
       dtype=torch.float64)
Class indices: tensor([6., 0., 0., 0., 0., 0., 0.], device='cuda:0', dtype=torch.float64)
Class range: 0.0 - 6.0
Original class IDs: tensor([6., 0., 0., 0., 0., 0., 0.], device='cuda:0', dtype=torch.float64)
Mapped class IDs: tensor([ 5., -1., -1., -1., -1., -1., -1.], device='cuda:0',
       dtype=torch.float64)
Mapped class range: -1.0 - 5.0
Final class indices: tensor([ 5., -1., -1., -1., -1., -1., -1.], device='cuda:0',
       dtype=torch.float64)
Final class range

 10%|█         | 2/20 [00:00<00:05,  3.13it/s]

...

[]
Image ID: 885
Original targets shape: torch.Size([9, 6])
Targets data:
tensor([[0.0000e+00, 0.0000e+00, 4.3330e-01, 4.6248e-01, 2.1889e-01, 3.2534e-01],
        [0.0000e+00, 0.0000e+00, 4.3909e-01, 3.0595e-01, 1.7473e-01, 2.6492e-01],
        [0.0000e+00, 0.0000e+00, 9.3075e-01, 2.0591e-01, 6.7766e-02, 3.5544e-01],
        [0.0000e+00, 0.0000e+00, 6.7922e-01, 1.6614e-01, 5.1234e-02, 1.9797e-02],
        [0.0000e+00, 0.0000e+00, 4.4905e-01, 1.6592e-01, 6.9766e-02, 1.7906e-02],
        [0.0000e+00, 3.8000e+01, 6.2506e-01, 5.8533e-01, 1.2709e-01, 6.2844e-02],
        [0.0000e+00, 0.0000e+00, 8.4669e-01, 1.6833e-01, 1.0287e-01, 1.5813e-02],
        [0.0000e+00, 0.0000e+00, 1.7344e-03, 1.6650e-01, 9.2094e-02, 1.3891e-02],
        [0.0000e+00, 0.0000e+00, 7.8042e-01, 1.6681e-01, 1.1745e-01, 2.0906e-02]],
       device='cuda:0', dtype=torch.float64)
Class indices: tensor([ 0.,  0.,  0.,  0.,  0., 38.,  0.,  0.,  0.], device='cuda:0',
       dtype=torch.float64)
Class range: 0.0 - 38.0
Orig

 95%|█████████▌| 19/20 [00:05<00:00,  3.62it/s]

[[0.90681416 0.21084261 0.10401594 0.33655357 0.5445481  0.        ]
 [0.43810704 0.29189208 0.22918247 0.38984907 0.39916506 0.        ]
 [0.68194115 0.15973718 0.04494476 0.03083773 0.32714316 0.        ]]
Image ID: 1296
Original targets shape: torch.Size([4, 6])
Targets data:
tensor([[0.0000e+00, 6.7000e+01, 6.3353e-01, 2.2403e-01, 1.1206e-01, 2.1141e-01],
        [0.0000e+00, 7.4000e+01, 7.4614e-01, 6.3791e-01, 2.7391e-02, 2.5953e-02],
        [0.0000e+00, 0.0000e+00, 1.7681e-01, 2.4531e-03, 6.5600e-01, 9.9105e-01],
        [0.0000e+00, 0.0000e+00, 5.7600e-01, 3.9562e-02, 2.2962e-01, 3.3333e-01]],
       device='cuda:0', dtype=torch.float64)
Class indices: tensor([67., 74.,  0.,  0.], device='cuda:0', dtype=torch.float64)
Class range: 0.0 - 74.0
Original class IDs: tensor([67., 74.,  0.,  0.], device='cuda:0', dtype=torch.float64)
Mapped class IDs: tensor([66., 73., -1., -1.], device='cuda:0', dtype=torch.float64)
Mapped class range: -1.0 - 73.0
Final class indices: tensor([66., 73

100%|██████████| 20/20 [00:05<00:00,  3.57it/s]

[[0.1678063  0.01437345 0.6299232  0.96934694 0.6257987  0.        ]
 [0.16397819 0.32936952 0.23045015 0.6331632  0.30293116 0.        ]]


In [15]:
# predictionsBefore = []
# predictionsAfter = []
# lossesBefore = []
# lossesAfter = []
# # mode = "image" # need different modes if i want to save image or output prediction json
# mode = "json"
# # image_ids= [71711,19221,22192] # output images that i want, 19221 is broccoli, 22192 is dog, 71711 is plane
# # image_ids= [139, 285, 632, 724, 776, 785, 802, 872, 885, 1000,
# #             1268, 1296, 1353,1425, 1490, 1503, 1532, 1584, 1675, 1761] # sample image id
# image_ids = [139]

# os.makedirs("./data/results/images", exist_ok=True)

# for i, (images, targets) in enumerate(tqdm(val_loader)):
#     if targets[0].numel() != 0:
#         with torch.no_grad():
#             #* modify inputs to be in proper shape
#             images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
#             images = images.to(device)
#             image_id = int(targets[0][0,0].cpu().numpy()) # assume 1 image
#             if image_id not in image_ids: continue # for when we want outputs of specific images
#             for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
#                 if boxes.ndim == 2: boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss. this is normally done in ListDataset -> collate_fn. the id now starts at 0 for each image
#             targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
#             # originalImageSize = targets[0, 6:].cpu().numpy() # original image shape, assume one image per batch - NOT available in json format
#             img_info = coco_dataset_val.coco.imgs[image_id]
#             originalImageSize = (img_info['height'], img_info['width'])
#             targets = targets[:, :6]

#             #* loss
#             model.train()
#             # start = time.time()
#             outputsBefore = model(images)
#             # end = time.time()
#             # print(end - start)
#             lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
#             lossesBefore.append(lossBefore.cpu().numpy())

#             images_adv = attacker.forward(images, targets) # get adversarial image

#             outputsAfter = model(images_adv)
#             lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
#             lossesAfter.append(lossAfter.cpu().numpy())

#             #* plot
#             model.eval()

#             # ground truth
#             # print(targets) #(ima ge,class,x,y,w,h), the class id starts from 1
#             # nms is (x1, y1, x2, y2, conf, cls), the class id starts from 0
#             # yolo is (x_center, y_center, width, height, conf. cls)

#             # before attack
#             outputsBefore = model(images[0].unsqueeze(0))
#             boxesBefore = non_max_suppression(outputsBefore, conf_thres=0.3, iou_thres=0.5)[0].numpy()
#             if mode == "json":
#                 boxesBefore = rescale_boxes(boxesBefore, img_size, originalImageSize)
#             boxesBefore = nms2yolo(boxesBefore, images)
#             if mode == "image":
#                 saveImageWithBoxes(images[0], boxesBefore, class_names, f"./data/results/images/attack_before_{image_id}.jpg")
#             if mode == "json":
#                 predictionsBefore += yolo2json(boxesBefore, images[0].unsqueeze(0), image_id)

#             # after attack
#             outputsAfter = model(images_adv[0].unsqueeze(0))
#             boxesAfter = non_max_suppression(outputsAfter, conf_thres=0.3, iou_thres=0.5)[0].numpy()


#             if mode == "json":
#                 boxesAfter = rescale_boxes(boxesAfter, img_size, originalImageSize)
#             # print(boxesAfter)
#             boxesAfter = nms2yolo(boxesAfter, images_adv)
#             print(boxesAfter)
#             if mode == "image":
#                 saveImageWithBoxes(images_adv[0], boxesAfter, class_names, f"./data/results/images/attack_after_{image_id}.jpg")

#                 # attackImage = images_adv[0] # for saving the same attack image for different pruning ratios, comment out after save
#                 # saveImageWithBoxes(attackImage, boxesAfter, class_names, f"./data/results/images/pruning/{image_id}/attack_after_99_x.jpg") # plot different pruning ratios with same attack image

#                 # greyscaleAttackImage = imgToGreyscale(attackImage)
#                 # saveImageWithBoxes(greyscaleAttackImage, boxesAfter, class_names, f"./data/results/images/pruning/{image_id}/attack_after_x_grey.jpg") # plot different pruning ratios with same attack image
#             if mode == "json":
#                 predictionsAfter += yolo2json(boxesAfter, images_adv[0].unsqueeze(0), image_id)
#             # time.sleep(0.1) # for using noise attack

#     else: continue # pics without targets
#     # break


# with open(f'./data/results/predictionsBefore.json', 'w') as f:
#     json.dump(predictionsBefore, f)
# with open(f'./data/results/predictionsAfter.json', 'w') as f:
#     json.dump(predictionsAfter, f)
# np.savetxt("./data/results/lossesBefore.csv", lossesBefore, delimiter=",")
# np.savetxt("./data/results/lossesAfter.csv", lossesAfter, delimiter=",")

In [16]:
# data = np.loadtxt('./data/results/lossesBefore.csv', delimiter=',')
# average = np.mean(data)
# print("Avg loss before attack:", average)
# data = np.loadtxt('./data/results/lossesAfter.csv', delimiter=',')
# average = np.mean(data)
# print("Avg loss after attack:", average)

# Get mAP

In [17]:
# from pycocotools.coco import COCO
# from pycocotools.cocoeval import COCOeval

# coco_gld = COCO(annFile_val) # coco
# # if modelv == 2:
# #     coco_rst = coco_gld.loadRes('./data/results/v2predictions.json')
# # elif modelv == 3:
# #     coco_rst = coco_gld.loadRes('./data/results/v3predictions.json')

# coco_rst = coco_gld.loadRes('./data/results/predictionsAfter.json')
# cocoEval = COCOeval(coco_gld, coco_rst, iouType='bbox')
# cocoEval.evaluate()
# cocoEval.accumulate()
# cocoEval.summarize()